# NB_patient_360

Build `adb_caresync.gold.patient_360` from the STTM workbook.

* Target grain: one row per patient
* Driving table: `adb_caresync.silver.patients`
* Required joins: `adb_caresync.silver.hospitals`, `adb_caresync.silver.insurance_providers`, and aggregated `adb_caresync.silver.lab_results`
* Load type: full rebuild with `CREATE OR REPLACE TABLE`
* Verified schema adjustments from source inspection:
  * `patients.date_of_birth` is a `TIMESTAMP`, so this notebook casts it to `DATE`
  * all referenced join keys and aggregation columns exist in the verified silver schemas


In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360 AS
WITH lab_results_agg AS (
  SELECT
    patient_id,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE) AS date_of_birth,
  CASE
    WHEN p.date_of_birth IS NULL THEN NULL
    ELSE CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT)
  END AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name AS insurance_provider_name,
  COALESCE(lra.total_tests, 0) AS total_tests,
  COALESCE(lra.abnormal_test_count, 0) AS abnormal_test_count,
  COALESCE(lra.critical_test_count, 0) AS critical_test_count,
  lra.last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_results_agg lra
  ON p.patient_id = lra.patient_id;

In [0]:
processed_rows = spark.table("adb_caresync.gold.patient_360").count()
print(f"processed_rows={processed_rows}")
dbutils.notebook.exit(str(processed_rows))